# Fraud / Anomaly Detection
## Notebook 06 — Statistical Anomaly Detection

### Objective
Build transparent Z-score, IQR, and Mahalanobis baselines.

## Imports and data

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.fraud_detection import load_features_labels, evaluate_scores, save_results
X, y = load_features_labels(ROOT)
RANDOM_STATE = 42
print('Features:', X.shape, '| fraud:', int(y.sum()))
from scipy.stats import chi2
from sklearn.covariance import LedoitWolf

**Interpretation.** Statistical rules provide explainable reference baselines.

**ML decision.** Evaluate their continuous severity scores, not only binary flags.

## Z-score and IQR

In [ ]:
amount = X['Amount'].to_numpy()
z_score = np.abs((amount - amount.mean()) / amount.std())
q1, q3 = np.quantile(amount, [.25, .75]); iqr = q3-q1
iqr_score = np.maximum((q1-amount)/iqr, (amount-q3)/iqr)
pd.DataFrame([evaluate_scores(y, z_score, 'Z-Score'), evaluate_scores(y, iqr_score, 'IQR')])

**Interpretation.** These univariate baselines detect unusual amounts but miss multivariate fraud patterns.

**ML decision.** Keep them as interpretable baselines.

## Mahalanobis distance

In [ ]:
cols = [c for c in X.columns if c.startswith('V')]
sample = X[cols].sample(min(50000, len(X)), random_state=RANDOM_STATE)
cov = LedoitWolf().fit(sample)
mahal = cov.mahalanobis(X[cols])
results = pd.DataFrame([evaluate_scores(y, z_score, 'Z-Score'), evaluate_scores(y, iqr_score, 'IQR'), evaluate_scores(y, mahal, 'Mahalanobis')])
save_results(results, ROOT/'reports/statistical_results.csv'); results

**Interpretation.** Shrinkage covariance stabilizes Mahalanobis distance in correlated high-dimensional data.

**ML decision.** Pass the strongest statistical baseline to the comparison stage.